ルールベースで提出用ファイルを作成する。

In [ ]:
import pandas as pd


In [3]:
import pandas as pd

transactions = pd.read_csv(
    # "C:/kaggle_nvidia/kaggle_hm/competitions/hm/data/processed/transactions_with_week.csv",
    "/workspace/data/processed/transactions_with_week.csv",
    dtype={
        "article_id": "string",
        "week": "int8",
        "price": "float32",
        "sales_channel_id": "int8"
    }
)

customers = pd.read_csv(
    # "C:/kaggle_nvidia/kaggle_hm/competitions/hm/data/processed/customers_converted.csv"
    "/workspace/data/processed/customers_converted.csv"
)

print(f"読み込み完了: {len(transactions)}行")





読み込み完了: 31788324行


transaction最終週の人気上位12件

In [7]:
# transactionsの最終週の人気top12を取得
last_week = transactions["week"].max()
print(f"最終週の週番号: {last_week}")

top12 = (
    transactions[transactions["week"] == last_week]
    .groupby("article_id")["customer_id"]
    .count()
    .reset_index()
    .rename(columns={"customer_id": "count"})
    .sort_values("count", ascending=False)
    .head(12)
)

print(top12)
# 全ユーザーに同じtop12を推薦
top12_article_ids = " ".join(top12["article_id"].astype(str).values)
print(f"推薦商品: {top12_article_ids}")



submission = pd.DataFrame({
    "customer_id": customers["customer_id"],
    "prediction": top12_article_ids
})

print(f"提出ファイル行数: {len(submission)}")
print(submission.head())


# 保存
submission.to_csv(
    # "C:/kaggle_nvidia/kaggle_hm/competitions/hm/outputs/submissions/submission_top12.csv",
    "/workspace/data/processed/sub_top12latest.csv",
    index=False
)
print("保存完了")

最終週の週番号: 104
       article_id  count
16594  0924243001    716
16595  0924243002    570
16253  0918522001    542
16574  0923758001    501
11287  0866731001    481
15572  0909370001    476
4292   0751471001    454
16068  0915529005    437
16067  0915529003    435
418    0448509014    430
3294   0714790020    430
4803   0762846027    401
推薦商品: 0924243001 0924243002 0918522001 0923758001 0866731001 0909370001 0751471001 0915529005 0915529003 0448509014 0714790020 0762846027
提出ファイル行数: 1371980
   customer_id                                         prediction
0            1  0924243001 0924243002 0918522001 0923758001 08...
1            2  0924243001 0924243002 0918522001 0923758001 08...
2            3  0924243001 0924243002 0918522001 0923758001 08...
3            4  0924243001 0924243002 0918522001 0923758001 08...
4            5  0924243001 0924243002 0918522001 0923758001 08...
保存完了


In [8]:
transactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id,week
0,2018-09-20,3,0663713001,0.050831,2,0
1,2018-09-20,3,0541518023,0.030492,2,0
2,2018-09-20,8,0505221004,0.015237,2,0
3,2018-09-20,8,0685687003,0.016932,2,0
4,2018-09-20,8,0685687004,0.016932,2,0


直近の直近の購買カテゴリに沿った候補リストを作成

In [10]:
import pandas as pd
from pathlib import Path

DATA_DIR = (Path.cwd() / '../../data').resolve()

articles = pd.read_csv(DATA_DIR / 'raw/articles.csv', dtype={"article_id" :"string" }, usecols=['article_id', 'index_group_name'])
transactions = pd.read_csv(DATA_DIR / 'processed/transactions_with_week.csv', dtype={"article_id" :"string" }, usecols=['customer_id', 'article_id', 'week'])
customers = pd.read_csv(DATA_DIR / 'processed/customers_converted.csv', usecols=['customer_id'])

# 過去3ヶ月（12週間）
max_week = transactions['week'].max()
recent = transactions[transactions['week'] >= max_week - 11]
recent = recent.merge(articles, on='article_id', how='left')

# カテゴリの優先順位
priority = {
    'Ladieswear': 0,
    'Divided': 1,
    'Menswear': 2,
    'Sport': 3,
    'Baby/Children': 4,
}

# カテゴリごとの人気上位12件の候補リストを作成（過去1ヶ月）
one_month = transactions[transactions['week'] >= max_week - 3]
one_month = one_month.merge(articles, on='article_id', how='left')

category_top12 = {}
for category in priority.keys():
    top12 = (
        one_month[one_month['index_group_name'] == category]
        .groupby('article_id')
        .size()
        .sort_values(ascending=False)
        .head(12)
        .index.tolist()
    )
    category_top12[category] = top12

# 過去3ヶ月に購買があるユーザーの最新weekを取得
user_latest_week = recent.groupby('customer_id')['week'].max().reset_index(name='latest_week')

# 最新weekのトランザクションだけ抽出
latest_transactions = recent.merge(user_latest_week, on='customer_id')
latest_transactions = latest_transactions[latest_transactions['week'] == latest_transactions['latest_week']]

# 優先順位が最も高いカテゴリを選択
latest_transactions['priority'] = latest_transactions['index_group_name'].map(priority)
latest_transactions = latest_transactions.dropna(subset=['priority'])
latest_transactions['priority'] = latest_transactions['priority'].astype(int)

user_category = (
    latest_transactions.groupby('customer_id')
    .apply(lambda x: x.loc[x['priority'].idxmin(), 'index_group_name'])
    .reset_index(name='recommend_category')
)

# 全ユーザーにマージ（購買なしはNaN）
all_users = customers.merge(user_category, on='customer_id', how='left')

# 購買なしユーザーにはLadieswearを割り当て
all_users['recommend_category'] = all_users['recommend_category'].fillna('Ladieswear')

# レコメンドリストを割り当て
all_users['recommendation'] = all_users['recommend_category'].map(category_top12)

print(all_users.head(10))
print(f"\n全ユーザー数: {len(all_users)}")
print(f"\nカテゴリ分布:")
print(all_users['recommend_category'].value_counts())

   customer_id recommend_category  \
0            1         Ladieswear   
1            2            Divided   
2            3              Sport   
3            4         Ladieswear   
4            5         Ladieswear   
5            6         Ladieswear   
6            7            Divided   
7            8         Ladieswear   
8            9         Ladieswear   
9           10         Ladieswear   

                                      recommendation  
0  [0751471001, 0915526001, 0915529003, 075147104...  
1  [0706016001, 0448509014, 0714790020, 071562400...  
2  [0918292001, 0852584001, 0866731001, 090595700...  
3  [0751471001, 0915526001, 0915529003, 075147104...  
4  [0751471001, 0915526001, 0915529003, 075147104...  
5  [0751471001, 0915526001, 0915529003, 075147104...  
6  [0706016001, 0448509014, 0714790020, 071562400...  
7  [0751471001, 0915526001, 0915529003, 075147104...  
8  [0751471001, 0915526001, 0915529003, 075147104...  
9  [0751471001, 0915526001, 0915529003, 07

In [ ]:
# リストをスペース区切りの文字列に変換
all_users['prediction'] = all_users['recommendation'].apply(lambda x: ' '.join(str(i) for i in x))

# 提出ファイルを作成
submission = all_users[['customer_id', 'prediction']]

# 保存
OUTPUT_DIR = (Path.cwd() / '../../data/processed').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_DIR / 'sub_indexgroup.csv', index=False)

print(submission.head(5))
print(f"\n提出ファイル保存完了: {OUTPUT_DIR / 'sub_indexgroup.csv'}")

   customer_id                                         prediction
0            1  0751471001 0915526001 0915529003 0751471043 08...
1            2  0706016001 0448509014 0714790020 0715624001 07...
2            3  0918292001 0852584001 0866731001 0905957001 09...
3            4  0751471001 0915526001 0915529003 0751471043 08...
4            5  0751471001 0915526001 0915529003 0751471043 08...

提出ファイル保存完了: /workspace/data/processed/sub_indexgroup.csv


: 